Get the k-fold and fam-fold partitions from https://github.com/marcellszi/dl-rna/releases

Paper:
  https://doi.org/10.1093/bioinformatics/btac415
  https://github.com/marcellszi/dl-rna

In [ ]:
import os
import glob
import hashlib
import tarfile
import urllib.request
import pandas as pd

In [ ]:
SPLITS_URL = "https://github.com/marcellszi/dl-rna/releases/download/Data/ct-splits.tar.gz"
SPLITS_SHA256 = "0ff7209016ba1775288572794e850d5857937c391d67c9ccacfe9ae77c0188e5"
DOWNLOAD_DIR = "./dl-rna/"
DATA_DIR = DOWNLOAD_DIR + "ct/"
OUT_DIR = "./"  # the files in the repo are in ../../../data/archiveII/
MAX_LEN = 512

if not os.path.isdir(DATA_DIR):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    tar_path = DOWNLOAD_DIR + "ct-splits.tar.gz"
    urllib.request.urlretrieve(SPLITS_URL, tar_path)
    with open(tar_path, "rb") as f:
        assert hashlib.sha256(f.read()).hexdigest() == SPLITS_SHA256, "ct-splits.tar.gz does not match the release used"
    with tarfile.open(tar_path) as tar:
        tar.extractall(DOWNLOAD_DIR, **({"filter": "data"} if hasattr(tarfile, "data_filter") else {}))

In [ ]:
# fam-fold numbers used in data/archiveII/fam_folds_split.csv
FAM_FOLD_NUMBERS = {"grp1": 0, "tmRNA": 1, "tRNA": 2, "5s": 3, "srp": 4,
                    "telomerase": 5, "RNaseP": 6, "16s": 7, "23s": 8}

rows = []
for fname in sorted(glob.glob(DATA_DIR + "*/*/*/*.ct")):
    split_type, fold_name, partition, ct_file = os.path.relpath(fname, DATA_DIR).split(os.sep)
    with open(fname) as f:
        seq_len = int(f.readline().split()[0])
    if seq_len >= MAX_LEN:
        continue
    fold_number = FAM_FOLD_NUMBERS[fold_name] if split_type == "fam-fold" else int(fold_name)
    rows.append([split_type, fold_name, fold_number, partition, ct_file[:-3]])

all_splits = pd.DataFrame(rows, columns=["split_type", "fold_name", "fold_number", "partition", "id"])
all_splits = all_splits.sort_values(["split_type", "fold_number", "partition", "id"], ignore_index=True)
all_splits.groupby(["split_type", "fold_number", "fold_name"]).size()

In [ ]:
kfold = all_splits[all_splits.split_type == "k-fold"][["fold_number", "partition", "id"]]
famfold = all_splits[all_splits.split_type == "fam-fold"][["fold_name", "fold_number", "partition", "id"]]
kfold.to_csv(OUT_DIR + "random_kfolds_split.csv", index=False)
famfold.to_csv(OUT_DIR + "fam_folds_split.csv", index=False)

In [ ]:
# check against the splits in the repo, when available
REPO_DIR = "../../../data/archiveII/"
for name, df in [("random_kfolds_split.csv", kfold), ("fam_folds_split.csv", famfold)]:
    if os.path.isfile(REPO_DIR + name):
        repo = pd.read_csv(REPO_DIR + name)
        same = df.merge(repo, how="outer", indicator=True)._merge.eq("both").all()
        print(name, "same rows as repo:", same, f"({len(df)} rows)")